# Aurora: control vs. K&C bogus for Typhoon Mawar (WP02 2023)

Runs Microsoft Aurora (0.1° fine-tuned, `AuroraHighRes`) twice from the same
IFS analysis at **2023-05-26 06Z**, about a day before Mawar's peak:

1. **control**: the unaltered analysis;
2. **bogus**: the analysis with a Kwon & Cheong (2009) vortex matched to the
   JTWC best-track intensity.

Both runs go 72 h in 6 h steps. At each step the storm is located by
`tcinit.rollout_tracking.track_rollout`, which centres each box on the previous
step's position, and the two runs are compared against the best track.

**Run on:** a cee-compute GPU node in `torch-venv` (see `envs/README.md`), with
`HF_HOME` pointing at scratch.

**Time ordering.** Aurora's batch history axis is oldest first: `[t − 6 h, t]`,
with the current state last. The earlier CausalTC notebook
(`physics_bogus_aurora.ipynb`) loaded `[06Z, 00Z]`, the reverse, so the old
`aurora_rollout_*` results are not directly comparable to this notebook. Cell 3
checks the ordering against the best track before anything runs.

## 1. Configuration

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

# Pick one A100. Must be set before torch is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

PROJECT = Path("/projects/aurora")
BEST_TRACK_FILE = PROJECT / "best-tracks/westpac/bwp2023/bwp022023.dat"
IFS_DIR = PROJECT / "ecmwf_ifs_01deg"
MODEL_CKPT = (
    PROJECT
    / "aurora_models/models--microsoft--aurora/snapshots"
    / "d8753da77552eab0f27150c0efb91cb3acb425d5/aurora-0.1-finetuned.ckpt"
)
STATIC_CACHE = PROJECT / "aurora_data"
OUT_DIR = Path(f"/fgs/{os.environ['USER']}/causaltc/demo_out/aurora_mawar")

START_TIME = datetime(2023, 5, 26, 6)
STEP = timedelta(hours=6)
N_STEPS = 12  # 72 h
BOX_HALF_DEG = 10.0
LEVELS = (1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50)

for p in (BEST_TRACK_FILE, IFS_DIR, MODEL_CKPT):
    if not p.exists():
        raise FileNotFoundError(p)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output -> {OUT_DIR}")

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xarray as xr
from aurora import AuroraHighRes, Batch, Metadata, rollout
from huggingface_hub import hf_hub_download

from tcinit import BestTrack
from tcinit.backends import aurora_backend
from tcinit.constants import HPA_TO_PA
from tcinit.plotting import plot_snapshot
from tcinit.rollout_snapshots import plot_rollout_comparison, plot_rollout_snapshots
from tcinit.rollout_tracking import build_bogus, snapshot_from_box, track_rollout

print("torch", torch.__version__, "on", torch.cuda.get_device_name(0))

## 2. Best track

In [ ]:
bt = BestTrack(BEST_TRACK_FILE)
bt_center = bt.get_location(START_TIME)
print(
    f"{bt.ID} at {START_TIME:%Y-%m-%d %HZ}: "
    f"({bt_center[0]:.1f}N, {bt_center[1]:.1f}E), "
    f"{bt.get_windspeed(START_TIME):.0f} m/s, "
    f"{bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa"
)

## 3. Initial batch from the 0.1° IFS analysis

Aurora takes two time slots, `[t − 6 h, t]`. Each is read by valid time,
so a 00Z start (whose t − 6 h slot is in the previous day's file) works too.
The check at the end extracts the storm from each slot and compares its
position with the best track at that slot's time. If the slots were swapped,
the positions would match the wrong times.

In [ ]:
HISTORY = (START_TIME - STEP, START_TIME)  # oldest first, current last
SURF = {"2t": "t2m", "10u": "u10", "10v": "v10", "msl": "msl"}  # Aurora -> GRIB name
ATMOS = ("t", "u", "v", "q", "z")

_open = {}


def _grib(t, kind):
    path = IFS_DIR / (
        f"hres_surf_{t:%Y%m%d}.grb"
        if kind == "surf"
        else f"hres_atmos_{t:%Y%m%d}_fh00_06_12_18.grb"
    )
    if path not in _open:
        _open[path] = xr.open_dataset(
            path, engine="cfgrib", backend_kwargs={"indexpath": ""}
        )
    return _open[path]


def _load(var, kind):
    # (batch=1, time, [level,] lat, lon) float32, time slots in HISTORY order.
    frames = []
    for t in HISTORY:
        da = _grib(t, kind)[var].sel(time=np.datetime64(t))
        if kind == "atmos":
            da = da.sel(isobaricInhPa=list(LEVELS)).transpose(
                "isobaricInhPa", "latitude", "longitude"
            )
        frames.append(da.values.astype(np.float32))
    return torch.from_numpy(np.stack(frames)[None])


ref = _grib(START_TIME, "atmos")
batch = Batch(
    surf_vars={k: _load(v, "surf") for k, v in SURF.items()},
    static_vars={},
    atmos_vars={k: _load(k, "atmos") for k in ATMOS},
    metadata=Metadata(
        lat=torch.from_numpy(ref.latitude.values.astype(np.float32)),
        lon=torch.from_numpy(ref.longitude.values.astype(np.float32)),
        time=(START_TIME,),
        atmos_levels=LEVELS,
    ),
)
for ds in _open.values():
    ds.close()

batch = batch.regrid(res=0.1)
static_path = hf_hub_download(
    repo_id="microsoft/aurora",
    filename="aurora-0.1-static.pickle",
    cache_dir=STATIC_CACHE,
)
with open(static_path, "rb") as f:
    batch.static_vars = {
        k: torch.from_numpy(np.asarray(v, dtype=np.float32))
        for k, v in pickle.load(f).items()
    }

print("time:", batch.metadata.time, " levels:", batch.metadata.atmos_levels)
for k, v in {**batch.surf_vars, **batch.atmos_vars}.items():
    print(f"  {k:4s} {tuple(v.shape)}")

In [ ]:
# Ordering check: slot i should hold the storm at HISTORY[i].
for i, t in enumerate(HISTORY):
    center = bt.get_location(t)
    box = aurora_backend.extract_snapshot_dataset(
        batch, box_center=center, box_size=BOX_HALF_DEG, time_idx=i
    )
    s = snapshot_from_box(box, valid_time=t, search_center=center)
    print(
        f"slot {i} ({t:%d %HZ}): model storm at ({s.loc[0]:.2f}, {s.loc[1]:.2f}), "
        f"{s.mslp_min / HPA_TO_PA:.1f} hPa | best track at ({center[0]:.2f}, {center[1]:.2f})"
    )

## 4. Control initial state

`time_idx=-1` is the current slot. `aurora_backend.extract_snapshot_dataset`
defaults to `time_idx=0`, which in a correctly ordered batch is t − 6 h.

In [ ]:
def extract_box(state, center):
    return aurora_backend.extract_snapshot_dataset(
        state, box_center=center, box_size=BOX_HALF_DEG, time_idx=-1
    )


box_ctrl0 = extract_box(batch, bt_center)
snap_ctrl0 = snapshot_from_box(box_ctrl0, valid_time=START_TIME, search_center=bt_center)
plot_snapshot(
    snap_ctrl0,
    box_ctrl0,
    title=f"Mawar, IFS analysis {START_TIME:%Y-%m-%d %HZ} (control initial state)",
)
plt.show()

## 5. Bogus initial state

`build_bogus` detects the storm, builds the K&C vortex from the best-track
intensity, and applies it to the box. `aurora_backend.write_back` then writes
the box into **both** time slots. This matches the CausalTC prototype, but it
means that inside the box the bogus run's t − 6 h slot equals its current slot.

In [ ]:
def clone_batch(b):
    md_ = b.metadata
    return Batch(
        surf_vars={k: v.clone() for k, v in b.surf_vars.items()},
        static_vars={k: v.clone() for k, v in b.static_vars.items()},
        atmos_vars={k: v.clone() for k, v in b.atmos_vars.items()},
        metadata=Metadata(
            lat=md_.lat.clone(),
            lon=md_.lon.clone(),
            time=tuple(md_.time),
            atmos_levels=tuple(md_.atmos_levels),
            rollout_step=md_.rollout_step,
        ),
    )


batch_bogus = clone_batch(batch)
_, bv, box_bogus0 = build_bogus(box_ctrl0, bt, START_TIME, search_center=bt_center)
aurora_backend.write_back(
    batch_bogus,
    box_bogus0,
    box_lats=box_ctrl0.latitude.values,
    box_lons=box_ctrl0.longitude.values,
)

print(f"p_c  = {bv.p_c / HPA_TO_PA:.1f} hPa (best track)")
print(f"V_m  = {bv.V_m:.1f} m/s")
print(f"R_v  = {bv.R_v_km:.0f} km")

box_check = extract_box(batch_bogus, bt_center)
snap_bogus0 = snapshot_from_box(box_check, valid_time=START_TIME, search_center=bt_center)
print(
    f"MSLP min: control {snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa -> "
    f"bogus {snap_bogus0.mslp_min / HPA_TO_PA:.1f} hPa"
)
plot_snapshot(
    snap_bogus0,
    box_check,
    title=f"Mawar, IFS analysis + K&C bogus {START_TIME:%Y-%m-%d %HZ}",
)
plt.show()

## 6. Load Aurora

In [ ]:
model = AuroraHighRes()
model.load_checkpoint_local(MODEL_CKPT, strict=False)
model.eval()
model = model.to("cuda")

## 7. Run and track both forecasts

`aurora_states` yields the initial batch, then each forecast step.
`track_rollout` consumes them one at a time and keeps only the storm box,
so no global field is held in memory beyond the current step.

In [ ]:
def aurora_states(b):
    yield b
    yield from rollout(model, b, steps=N_STEPS)


common = dict(
    cyclone_id=bt.ID, start_time=START_TIME, initial_center=bt_center, step=STEP
)
with torch.inference_mode():
    ctrl = track_rollout(aurora_states(batch), extract_box, **common)
    bogus = track_rollout(
        aurora_states(batch_bogus), extract_box, intervened=True, **common
    )

ctrl.to_netcdf(OUT_DIR / "control")
bogus.to_netcdf(OUT_DIR / "bogus")

## 8. Compare with the best track

In [ ]:
cols = {
    "model_lead_hours": "lead_h",
    "model_MSLP": "MSLP",
    "model_VMAX": "VMAX",
    "position_error_km": "pos_err_km",
}
df_ctrl = ctrl.to_comparison_dataframe(bt)
df_bogus = bogus.to_comparison_dataframe(bt)
table = pd.concat(
    {
        "control": df_ctrl[list(cols)].rename(columns=cols),
        "bogus": df_bogus[list(cols)].rename(columns=cols).drop(columns="lead_h"),
        "best track": df_ctrl[["bt_MSLP", "bt_VMAX"]].rename(
            columns={"bt_MSLP": "MSLP", "bt_VMAX": "VMAX"}
        ),
    },
    axis=1,
)
table.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for df, label, style in ((df_ctrl, "control", "o-"), (df_bogus, "bogus", "^--")):
    lead = df["model_lead_hours"]
    axes[0].plot(lead, df["model_MSLP"], style, label=label)
    axes[1].plot(lead, df["model_VMAX"], style, label=label)
    axes[2].plot(lead, df["position_error_km"], style, label=label)
axes[0].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_MSLP"], "k-", lw=2, label="best track")
axes[1].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_VMAX"], "k-", lw=2, label="best track")
for ax, ylabel in zip(axes, ("min MSLP (hPa)", "max 10 m wind (kt)", "position error (km)")):
    ax.set_xlabel("lead time (h)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"Aurora, Mawar from {START_TIME:%Y-%m-%d %HZ}")
plt.tight_layout()
plt.show()

In [ ]:
plot_rollout_comparison(
    ctrl,
    bogus,
    OUT_DIR / "comparison_aurora_mawar.pdf",
    best_track=bt,
    labels=("Aurora control", "Aurora + K&C bogus"),
)
plot_rollout_snapshots(ctrl, OUT_DIR / "rollout_aurora_control.pdf", best_track=bt)
plot_rollout_snapshots(bogus, OUT_DIR / "rollout_aurora_bogus.pdf", best_track=bt)
print(sorted(p.name for p in OUT_DIR.glob("*.pdf")))